# 08. Capstone Project: End-to-End PySpark Data Pipeline

This notebook ties together the key ideas from the previous notebooks into a realistic data engineering pipeline.

## Project objective
Build a retail sales ETL pipeline using PySpark.

## Pipeline stages
- read raw CSV data
- clean invalid values
- transform to analytics-ready schema
- aggregate KPIs
- save outputs to Parquet


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('CapstonePipeline').master('local[*]').getOrCreate()

raw = spark.createDataFrame([
    ('2024-01-01', 'A', 1200),
    ('2024-01-02', 'A', 1500),
    ('2024-01-01', 'B', 900),
    ('2024-01-03', 'B', 1700),
    ('2024-01-02', 'C', 800),
], ['date', 'region', 'sales'])

cleaned = raw.withColumn('sales', F.col('sales').cast('double'))
cleaned = cleaned.filter(F.col('sales').isNotNull())

summary = cleaned.groupBy('region').agg(
    F.sum('sales').alias('total_sales'),
    F.avg('sales').alias('avg_sales'),
    F.count('sales').alias('sales_records')
)

summary.show()
cleaned.write.mode('overwrite').parquet('data/output/capstone_sales_cleaned.parquet')
summary.write.mode('overwrite').parquet('data/output/capstone_sales_summary.parquet')

spark.stop()


## Reflection questions
- Which steps are transformations?
- Where would caching help?
- How would you handle larger files or skewed partitions?
- How would you adapt this to BigQuery or GCS in production?

## Suggested next steps
1. parse date columns into proper timestamp/date types
2. add product or customer dimension tables
3. compute monthly KPIs
4. convert the logic to a reusable PySpark job
5. orchestrate with Airflow or Cloud Composer
